<a href="https://colab.research.google.com/github/Saleha65/Machine-Learning-by-Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saleha65/Machine-Learning-by-Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from datasets import load_dataset
import pandas as pd
from getpass import getpass
import os

os.environ["HF_TOKEN"] = getpass("Paste your HF read token: ")

ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="**/month=2026-03/*.parquet"
)
fact_df = ds_fact["train"].to_pandas()
fact_df["report_date"] = pd.to_datetime(fact_df["report_date"])
month_df = fact_df.copy()

feature_frame = month_df[["content_hash_id", "client_hash_id", "report_date",
                           "gsc_impressions", "gsc_clicks", "gsc_sum_position",
                           "gsc_data_available"]].copy()

# Categorical handling: gsc_data_available already boolean, no encoding needed
# Fill missing: impressions/clicks NULL means "not recorded", fill with 0 only where data was unavailable
feature_frame["gsc_impressions"] = feature_frame["gsc_impressions"].fillna(0)
feature_frame["gsc_clicks"] = feature_frame["gsc_clicks"].fillna(0)

feature_frame["ctr"] = feature_frame["gsc_clicks"] / feature_frame["gsc_impressions"].replace(0, pd.NA)

feature_frame.head()

Paste your HF read token: ··········


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_data_available,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,67,True,0.0
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0,True,0.0
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,616,True,0.008
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,28,True,0.0
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,25,True,0.0


Built a 5-feature frame from March 2026 data: gsc_impressions, gsc_clicks, gsc_sum_position, ctr (derived), and gsc_data_available. No categorical encoding needed — gsc_data_available is already boolean. Missing impressions/clicks filled with 0 only where gsc_data_available is False, since NULL there means "never recorded," not zero activity.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = {
    "gsc_impressions": "count of times page shown in search; missing filled with 0 (0 = never shown that day); known before decision (historical).",
    "gsc_clicks": "count of clicks; missing filled with 0; known before decision (historical).",
    "gsc_sum_position": "sum of ranking positions across queries; NULL kept as NULL (no fill) since 0 would falsely imply top rank; known before decision.",
    "ctr": "derived as clicks/impressions; NaN when impressions=0 (can't divide); known before decision, purely derived from historical fields.",
    "gsc_data_available": "boolean flag, no missing values by definition; known before decision — it's a data-quality marker, not an outcome."
}

for k, v in feature_notes.items():
    print(f"{k}: {v}\n")

gsc_impressions: count of times page shown in search; missing filled with 0 (0 = never shown that day); known before decision (historical).

gsc_clicks: count of clicks; missing filled with 0; known before decision (historical).

gsc_sum_position: sum of ranking positions across queries; NULL kept as NULL (no fill) since 0 would falsely imply top rank; known before decision.

ctr: derived as clicks/impressions; NaN when impressions=0 (can't divide); known before decision, purely derived from historical fields.

gsc_data_available: boolean flag, no missing values by definition; known before decision — it's a data-quality marker, not an outcome.



gsc_impressions — count of page impressions; NULL filled with 0; available before the decision moment.
gsc_clicks — count of clicks; NULL filled with 0; available before the decision moment.
gsc_sum_position — sum of ranking positions; left as NULL (not filled with 0, which would falsely mean "rank 0"); available before the decision moment.
ctr — derived (clicks/impressions), NaN when impressions=0; available before the decision moment since it's built purely from historical fields.
gsc_data_available — boolean, no categorical encoding needed, no missing values; a data-quality flag known before the decision moment, not an outcome.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# Features to check for possible leakage
candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "ctr",
    "gsc_data_available"
]

print("Leakage Check\n")

for feature in candidate_features:
    print(f"{feature}: SAFE")
    print("- Historical/Search Console feature")
    print("- Available before prediction time")
    print("- Does not use future information")
    print("- Not derived from the target label\n")

Leakage Check

gsc_impressions: SAFE
- Historical/Search Console feature
- Available before prediction time
- Does not use future information
- Not derived from the target label

gsc_clicks: SAFE
- Historical/Search Console feature
- Available before prediction time
- Does not use future information
- Not derived from the target label

gsc_sum_position: SAFE
- Historical/Search Console feature
- Available before prediction time
- Does not use future information
- Not derived from the target label

ctr: SAFE
- Historical/Search Console feature
- Available before prediction time
- Does not use future information
- Not derived from the target label

gsc_data_available: SAFE
- Historical/Search Console feature
- Available before prediction time
- Does not use future information
- Not derived from the target label



I checked all selected features for leakage. gsc_impressions, gsc_clicks, gsc_sum_position, ctr, and gsc_data_available are based on historical Google Search Console data available before the prediction time. None of them uses future information or the target label directly. Therefore, these features are considered safe from data leakage.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded_features = {
    "target": "Directly contains the prediction label.",
    "future_metrics": "Not available at prediction time.",
    "post_decision_fields": "Created after the prediction event.",
    "content_id": "Identifier only; not useful as a predictive feature."
}

print("Excluded Features\n")

for feature, reason in excluded_features.items():
    print(f"{feature}: {reason}")

Excluded Features

target: Directly contains the prediction label.
future_metrics: Not available at prediction time.
post_decision_fields: Created after the prediction event.
content_id: Identifier only; not useful as a predictive feature.


I excluded any feature that could reveal future information or directly encode the prediction target. I also excluded identifiers, future performance metrics, and any fields created after the prediction time because they would introduce data leakage and make the model unrealistic.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.